In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import os
from sklearn.decomposition import PCA
from scipy.interpolate import interp1d
from scipy.stats import linregress
from matplotlib.lines import Line2D

filename = 'model_predicted_looming_responses.pkl'

# load predicted responses to looming stimuli
with open(filename, 'rb') as f:
    all_outputs = pickle.load(f)

# get response type labels (see Fig 2)
clusters = pd.read_csv('response_type_labels.csv')


In [ ]:
# select neurons/unit_ids present both in digital-twin and in the response type labels
stims = ['020', '032', '050', '080', '110']

neuron_ids_paths = ['data-quality/dynamic28188-19-9-Fluorescence-7b7_neurons_fluor_good.npy',
 'data-quality/dynamic28712-3-8-Fluorescence-7b7_neurons_fluor_good.npy',
 'data-quality/dynamic29163-4-4-Fluorescence-7b7_neurons_fluor_good.npy',
 'data-quality/dynamic29163-5-8-Fluorescence-7b7_neurons_fluor_good.npy',
 'data-quality/dynamic29163-6-5-Fluorescence-7b7_neurons_fluor_good.npy']

unit_ids = {}
for i, datakey in enumerate(clusters.datakey.unique()):
    print(datakey)
    unit_ids[datakey] = np.load(neuron_ids_paths[i]) + 1

def filter_data_by_unit_ids(data, unit_ids_dict, df, stims):
    filtered_data = {stim: {} for stim in stims}

    for datakey, unit_ids in unit_ids_dict.items():
        subset = df[df['datakey'] == datakey]
        if subset.empty:
            continue

        target_unit_ids = subset['unit_id'].values

        valid_unit_ids = np.intersect1d(unit_ids, target_unit_ids, assume_unique=True)

        if valid_unit_ids.size == 0:
            continue

        selected_indices = np.nonzero(np.isin(unit_ids, valid_unit_ids))[0]

        for stim in stims:
            if datakey not in data[stim]:
                continue
            arr = data[stim][datakey]
            filtered_data[stim][datakey] = arr[selected_indices]

    return filtered_data


all_outputs = filter_data_by_unit_ids(all_outputs, unit_ids, clusters, stims)

cluster_labels_df = clusters

### Plot mean responses of each response type

In [ ]:
second_keys = list(all_outputs[stims[0]].keys())

sampling_rate = 30.0   
plot_stims = ['020', '032', '050', '080', '110'] 
BASELINE_WINDOW = 3     # for subtracting baseline response (first 3 frames)

clustered_data = {}

for key in second_keys:
    subset_df = cluster_labels_df[cluster_labels_df['datakey'] == key]
        
    labels = subset_df['cluster_gmm'].values
    
    for stim in plot_stims:
        if stim in all_outputs and key in all_outputs[stim]:
            traces_matrix = all_outputs[stim][key]
            
            if traces_matrix.shape[0] != len(labels):
                continue
            
            for i, trace in enumerate(traces_matrix):
                cluster_id = labels[i]
                
                baseline_val = np.mean(trace[:BASELINE_WINDOW])
                corrected_trace = trace - baseline_val
                
                if cluster_id not in clustered_data:
                    clustered_data[cluster_id] = {}
                if stim not in clustered_data[cluster_id]:
                    clustered_data[cluster_id][stim] = []
                
                clustered_data[cluster_id][stim].append(corrected_trace)

# plotting
sorted_clusters = sorted(clustered_data.keys())
n_clusters = len(sorted_clusters)

fig, axes = plt.subplots(nrows=n_clusters, ncols=1, figsize=(10, 4 * n_clusters), sharex=False)
if n_clusters == 1: axes = [axes] 

colors = sns.color_palette("viridis", len(plot_stims))

for ax, cluster_id in zip(axes, sorted_clusters):
    ax.set_title(f"Cluster/ Response type {cluster_id}", fontsize=14)
    ax.set_ylabel("Mean response (baseline corrected)")
    
    for stim_idx, stim in enumerate(plot_stims):
        if stim in clustered_data[cluster_id]:
            trace_stack = np.array(clustered_data[cluster_id][stim])
            
            mean_trace = np.nanmean(trace_stack, axis=0)
            std_trace = np.nanstd(trace_stack, axis=0)
            
            time_axis = np.arange(len(mean_trace)) / sampling_rate
            
            ax.plot(time_axis, mean_trace, color=colors[stim_idx], label=f"Stim {stim}", linewidth=2)
            
            ax.fill_between(
                time_axis, 
                mean_trace - std_trace, 
                mean_trace + std_trace, 
                color=colors[stim_idx], 
                alpha=0.15
            )

    ax.grid(True, linestyle=':', alpha=0.6)

axes[-1].set_xlabel("Time (s)")
plt.tight_layout()
plt.show()

### Parametric analysis of l/v vs peak time relative to collision to get angular looming detection threshold

In [ ]:
# extrapolate traces until time of collision (TTC) for each stimulus, based on the rig geometry and looming stimulus parameters

sampling_rate = 30.0    # Hz
monitorHeight = 33      # cm
platformDistance = 15   # cm

lv_values = np.array([20, 32, 50, 80, 110]) 
lv_map = dict(zip(stims, lv_values))

theta_screen_deg = 2 * np.degrees(np.arctan((monitorHeight / 2) / platformDistance))
print(f"Max Screen Angle = {theta_screen_deg:.1f} degrees")

extra_frames_map = {}
for stim_label, lv_ms in zip(stims, lv_values):
    ttc_ms = lv_ms / np.tan(np.radians(theta_screen_deg / 2))
    ttc_s = ttc_ms / 1000.0
    frames = int(ttc_s * sampling_rate)
    extra_frames_map[stim_label] = frames

all_outputs_extended = {}

for stim in stims:
    all_outputs_extended[stim] = {}
    if stim not in all_outputs: continue
    keys = list(all_outputs[stim].keys())
    
    extra_frames = extra_frames_map[stim]
    
    for key in keys:
        raw_traces = all_outputs[stim][key]
        n_current = raw_traces.shape[1]
        x_current = np.arange(n_current)
        x_new = np.arange(n_current + extra_frames)
        f = interp1d(x_current, raw_traces, kind='linear', fill_value='extrapolate', axis=1)
        extended_traces = f(x_new)
        all_outputs_extended[stim][key] = extended_traces

BASELINE_WINDOW = 3 


clustered_traces = {} 
cluster_counts = {}

second_keys = list(all_outputs_extended[stims[0]].keys()) 

for stim in stims:
    for key in second_keys:
        if stim in all_outputs_extended and key in all_outputs_extended[stim]:
            traces = all_outputs_extended[stim][key]
            
            subset_df = cluster_labels_df[cluster_labels_df['datakey'] == key]
            if subset_df.empty or len(subset_df) != traces.shape[0]: continue
            current_labels = subset_df['cluster_gmm'].values 

            for i, cluster_id in enumerate(current_labels):
                if cluster_id not in clustered_traces:
                    clustered_traces[cluster_id] = {}
                if stim not in clustered_traces[cluster_id]:
                    clustered_traces[cluster_id][stim] = []
                
                raw_trace = traces[i]
                baseline = np.nanmean(raw_trace[:BASELINE_WINDOW]) # mean of first 3 frames
                corrected_trace = raw_trace - baseline
                
                clustered_traces[cluster_id][stim].append(corrected_trace)

for cid in clustered_traces:
    first_stim = list(clustered_traces[cid].keys())[0]
    cluster_counts[cid] = len(clustered_traces[cid][first_stim])

mean_peak_results = []
sorted_clusters = sorted(clustered_traces.keys())

for cluster_id in sorted_clusters:
    for stim in stims:
        if stim in clustered_traces[cluster_id]:
            trace_list = clustered_traces[cluster_id][stim]
            if not trace_list: continue
            
            mean_trace = np.nanmean(trace_list, axis=0)
            
            if cluster_id == 1:
                pass
            is_inverted = False
            if np.abs(np.min(mean_trace)) > np.max(mean_trace):
                is_inverted = True
                
            peak_idx = np.argmax(mean_trace)
            
            collision_idx = len(mean_trace) - 1
            frames_from_end = collision_idx - peak_idx
            time_from_end_ms = (frames_from_end / sampling_rate) * 1000.0
            
            mean_peak_results.append({
                'stimulus': stim,
                'l_v': lv_map[stim],
                'cluster': cluster_id,
                'time_to_collision_ms': time_from_end_ms,
                'is_inverted':is_inverted,
            })

df_means = pd.DataFrame(mean_peak_results)

In [ ]:
inversion_map = df_means.groupby('cluster')['is_inverted'].first().to_dict()

final_results = []
min_slope_threshold = 0.05

for cluster_id in sorted(clustered_traces.keys()):
    is_inv = inversion_map[cluster_id]
    
    x_lv = []
    y_ttc = []
    
    for stim in stims:
        if stim in clustered_traces[cluster_id]:
            traces = clustered_traces[cluster_id][stim]
            mean_trace = np.nanmean(traces, axis=0)
            
            if is_inv:
                mean_trace = np.abs(mean_trace)
            
            peak_idx = np.argmax(mean_trace)
            collision_idx = len(mean_trace) - 1
            frames_from_end = collision_idx - peak_idx
            time_from_end_ms = (frames_from_end / sampling_rate) * 1000.0
            
            x_lv.append(lv_map[stim])
            y_ttc.append(time_from_end_ms)
            
    if len(x_lv) >= 3:
        slope, intercept, r_val, p_val, std_err = linregress(x_lv, y_ttc)
        
        if slope > min_slope_threshold:
            theta_rad = 2 * np.arctan(1 / slope)
            theta_deg = np.degrees(theta_rad)
            delay_ms = abs(intercept)
            
            final_results.append({
                'cluster': cluster_id,
                'theta_threshold_deg': theta_deg,
                'delay_ms': delay_ms,
                'is_inverted': is_inv,
                'slope': slope,
                'intercept': intercept,
                'r_squared': r_val**2,
                'n_neurons': len(clustered_traces[cluster_id][stims[0]])
            })
        else:
             final_results.append({
                'cluster': cluster_id,
                'theta_threshold_deg': np.nan,
                'delay_ms': np.nan,
                'is_inverted': is_inv,
                'slope': slope,
                'intercept': intercept,
                'r_squared': r_val**2,
                'n_neurons': len(clustered_traces[cluster_id][stims[0]])
            })

df_results = pd.DataFrame(final_results)

df_results

In [ ]:
df_filtered = df_results[df_results['n_neurons'] >= 10].copy()
r2 = 0.95 # define a threshold for good fit

good_fits = df_filtered[
    (df_filtered['theta_threshold_deg'].notna()) & 
    (df_filtered.get('r_squared', 1.0) >= r2) 
]
max_theta = good_fits['theta_threshold_deg'].max() if not good_fits.empty else 10.0

dummy_height = max_theta 

def assign_group(row):
    if pd.isna(row['theta_threshold_deg']) or row.get('r_squared', 1.0) < r2:
        return 3  
    elif row.get('is_inverted', False):
        return 2  
    else:
        return 1  

df_filtered['plot_group'] = df_filtered.apply(assign_group, axis=1)

def assign_plot_height(row):
    if row['plot_group'] == 3:  
        return dummy_height
    else:
        return row['theta_threshold_deg']
        
df_filtered['plot_height'] = df_filtered.apply(assign_plot_height, axis=1)

df_plot = df_filtered.sort_values(by=['plot_group', 'plot_height', 'cluster'])

df_plot['cluster_str'] = df_plot['cluster'].astype(int).astype(str)

color_map = {
    1: 'gray',       # Normal
    2: 'orange',     # Inverted
    3: 'lightgray'   # NaNs
}
bar_colors = [color_map[group] for group in df_plot['plot_group']]



plt.figure(figsize=(2, 8)) 

ax = sns.barplot(
    data=df_plot, 
    x='plot_height', 
    y='cluster_str', 
    palette=bar_colors, 
    edgecolor='black', 
    alpha=0.8
)

x_offset = max_theta * 0.02 

for bar, (_, row) in zip(ax.patches, df_plot.iterrows()):
    width = bar.get_width()
    y_pos = bar.get_y() + bar.get_height() / 2
    
    if not np.isfinite(width) or not np.isfinite(y_pos):
        continue
        
    n_count = int(row['n_neurons'])
    text_label = f"{n_count}"
    
    ax.text(
        width + x_offset,  
        y_pos, 
        text_label, 
        ha='left', va='center', fontsize=9, color='black', rotation=0
    )

plt.xlabel(r"$\theta_{thres}$ (degrees)")
plt.ylabel("Cluster ID")
plt.xticks(np.arange(0, 20, 7.5))

plt.xlim(0, max_theta * 1.15) 

sns.despine()
plt.tight_layout()
plt.show()

### Linear fit of l/v vs peak time relative to collision 

In [ ]:
target_clusters = [16, 25, 49, 40] 

mean_peak_results = []
sorted_clusters = sorted(clustered_traces.keys())

for cluster_id in sorted_clusters:
    for stim in stims:
        if stim in clustered_traces[cluster_id]:
            trace_list = clustered_traces[cluster_id][stim]
            if not trace_list: continue
            
            mean_trace = np.nanmean(trace_list, axis=0)
            peak_idx = np.argmax(mean_trace)
            
            # Time to Collision
            collision_idx = len(mean_trace) - 1
            frames_from_end = collision_idx - peak_idx
            time_from_end_ms = (frames_from_end / sampling_rate) * 1000.0
            
            mean_peak_results.append({
                'stimulus': stim,
                'l_v': lv_map[stim],
                'cluster': cluster_id,
                'time_to_collision_ms': time_from_end_ms
            })

df_means = pd.DataFrame(mean_peak_results)


# plot
if df_means.empty:
    print("No data found.")
else:
    sns.set_style("ticks")
    inset_colors = sns.color_palette("viridis", len(stims))
    
    g = sns.lmplot(
        data=df_means,
        x='l_v',
        y='time_to_collision_ms',
        col='cluster',
        hue='cluster',
        col_wrap=4,
        height=3,      
        aspect=1.0,  
        scatter_kws={'alpha': 0.8, 's': 40, 'edgecolor': 'k', 'linewidth': 0.6},
        line_kws={'color': '#333333', 'linestyle': '--', 'alpha': 0.6, 'linewidth': 1.2},
        ci=None,
        col_order=target_clusters if target_clusters else None,
        palette='deep'
    )
    g.set(xlim=(15, 115))
    
    for ax in g.axes.flat:
        ax.grid(True, linestyle=':', alpha=0.4, color='gray')

        try:
            cluster_id = int(ax.get_title().split('=')[-1].strip())
        except:
            continue
            
        subset = df_means[df_means['cluster'] == cluster_id]
        n_members = cluster_counts.get(cluster_id, 0)
        
        # --- Regression Logic (Without Delay Text) ---
        title_str = ""
        if len(subset) >= 3:
            slope, intercept, r_val, p_val, std_err = linregress(subset['l_v'], subset['time_to_collision_ms'])
            if slope > 0.05:
                theta_deg = np.degrees(2 * np.arctan(1 / slope))
                # Removed the delay text as requested
                title_str = f"Cluster {cluster_id} (n={n_members})\n$\\theta_{{thres}} = {theta_deg:.1f}^\\circ$"
            else:
                title_str = f"Cluster {cluster_id} (n={n_members})\n(Flat/Inverted)"
        else:
            title_str = f"Cluster {cluster_id} (n={n_members})\n(Insufficient Data)"
        
        ax.set_title(title_str, fontsize=8)

        ax_ins = ax.inset_axes([0.05, 0.45, 0.55, 0.5]) 
        
        if cluster_id in clustered_traces:
            for i, stim in enumerate(stims):
                if stim in clustered_traces[cluster_id]:
                    traces = clustered_traces[cluster_id][stim]
                    mean_trace = np.nanmean(traces, axis=0)
                    time_axis = np.arange(len(mean_trace)) / sampling_rate
                    
                    ax_ins.plot(time_axis, mean_trace, color=inset_colors[i], linewidth=1.5, alpha=0.9)
        
        ax_ins.patch.set_alpha(0.0)
        ax_ins.axis('off')
    
    g.set_axis_labels("Stimulus $l/|v|$ (ms)", "Peak Time Relative to Collision(ms)")
    sns.despine(trim=False)
    
    legend_elements = [
        Line2D([0], [0], color=inset_colors[i], lw=2, label=f"{stims[i]}") 
        for i in range(len(stims))
    ]
    
    g.fig.legend(
        handles=legend_elements, 
        loc='upper center', 
        bbox_to_anchor=(0.5, 1.08), 
        ncol=len(stims),        
        frameon=False,
        fontsize=8
    )

    plt.tight_layout()
    plt.show()